# 3. Read current status

## Before you start

Use [the Ethernet guide](README.md) and the project's Python 3.12 kernel. Each
notebook runs independently against **shutter**, CH **0**, serial **68800777**,
firmware **1.0.7**, KEH3 middle bay at **192.168.1.19:40305**. Port **40303** is
hub management. Keep the hub powered, Ethernet connected and USB IN disconnected.
Close competing owners before connecting. No simulator, scan or USB fallback is used.

Construction is passive. The context releases communication on success or error;
it sends no output command. If release fails, retain `hub` and resolve the reported
error before retrying `hub.close()`. Restart the kernel after source changes.

In [2]:
import sys
from dataclasses import asdict
from pathlib import Path
from pprint import pprint

sys.path.insert(0, str(Path("docs/examples") if Path("docs/examples").is_dir() else Path.cwd()))
from shutter_config import SHUTTER  # noqa: E402

from thorlabs_shutter_control import ShutterHub  # noqa: E402

print(SHUTTER.name, SHUTTER.route.endpoint)

shutter 192.168.1.19:40305


`get_status()` explicitly requests identity, mode, interlock requirement and
status bits. It returns the same `ShutterStatus` schema used by the CLI and GUI.

| Field | Meaning |
| --- | --- |
| `identity` | Verified model, serial, firmware, hardware revision and channels |
| `feedback.operating_mode` | manual, single, auto or triggered |
| `feedback.electrical_state` | off/on from a fresh status-bit reply |
| `feedback.status_bits` | Raw 32-bit status value; output mask is `0x00000001` |
| `feedback.operating_state` | unknown; OFF does not establish Inactive or an unprimed trigger |
| `feedback.interlock_enabled` | Interlock requirement configuration, not contact continuity |
| `feedback.shutter_state` | unknown; there is no independent optical measurement in these queries |

The exact KSC101 .NET API defines solenoid feedback Closed=0/Open=1 and separate
OperatingStates Active=1/Inactive=2. C/older APT representations differ. Normal
readback uses the documented output bit and avoids the ambiguous scalar reply.
For a deliberate raw diagnostic, `hub.get_solenoid_state_reply(SHUTTER.name)`
returns the unmodified 0x04CD byte; it does not change normal state interpretation.

The prior real readback was OFF/Triggered with status bits `0x2000`; every run
requests current data. See [manual/API findings](../state-zero-investigation.md).

In [3]:
with ShutterHub([SHUTTER]) as hub:
    hub.connect(SHUTTER.name)
    status = hub.get_status(SHUTTER.name)
    pprint(asdict(status))

{'channel': 0,
 'cleanup': None,
 'connection': 'connected',
 'description': 'KSC101 firmware 1.0.7',
 'endpoint': '192.168.1.19:40305',
 'error': None,
 'feedback': {'channel_enabled': None,
              'electrical_observed_at': '2026-11-06T20:50:50.351326+00:00',
              'electrical_state': 'off',
              'interlock_enabled': True,
              'key_enabled': None,
              'operating_mode': 'triggered',
              'operating_state': 'unknown',
              'shutter_state': 'unknown',
              'status_bits': 8192},
 'feedback_freshness': 'response_after_request_no_transaction_id',
 'generation': 1,
 'has_device': True,
 'identity': {'channels': 1,
              'firmware': '1.0.7',
              'hardware_version': 3,
              'model': 'KSC101',
              'serial_number': '68800777'},
 'identity_verified': True,
 'name': 'shutter',
 'observed_at': '2026-11-06T20:50:50.351326+00:00',
 'observed_serial_number': '68800777',
 'optical_observed_at': N